# Test file for CIC-IDS2017 model creation and analysis

### 1. Data loading

In [4]:
import pandas as pd
import polars as pl
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta

import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

pd.options.display.max_rows = 80

In [5]:
# read data

#get dataset size for comparison
file_path = '/mnt/AI-DATA/alara/CiberIA_O1_A1/Data/UNSW-NB15.csv'
file_size = os.path.getsize(file_path)
print(f"Dataset size: {file_size / (1024 * 1024):.2f} MB")

start_time = datetime.now()
print("Reading data with Polars...")
df_pl = pl.read_csv(file_path, low_memory=False)
print("Data read using Polars in", datetime.now() - start_time) 

Dataset size: 1794.68 MB
Reading data with Polars...
Data read using Polars in 0:00:00.600488


In [6]:
start_time = datetime.now()
print("Reading data with Polars...")
df_pl = pl.read_csv(file_path, low_memory=False)
print("Data read using Polars in", datetime.now() - start_time) 

Reading data with Polars...
Data read using Polars in 0:00:00.550290


### 2. Data exploration

In [7]:
from IPython.display import display
#display(df_pd.head())
display(df_pl.head())

Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,…,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Fwd Segment Size Avg,Bwd Segment Size Avg,Fwd Bytes/Bulk Avg,Fwd Packet/Bulk Avg,Fwd Bulk Rate Avg,Bwd Bytes/Bulk Avg,Bwd Packet/Bulk Avg,Bwd Bulk Rate Avg,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
str,str,i64,str,i64,i64,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,…,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""175.45.176.2-149.171.126.16-23…","""175.45.176.2""",23357,"""149.171.126.16""",80,6,"""22/01/2015 07:50:15 AM""",214392,9,21,388.0,24564.0,194.0,0.0,43.111111,85.545959,1460.0,0.0,1169.714286,552.156965,116384.939737,139.930594,7392.827586,17881.622845,61855.0,2.0,213501.0,26687.625,35191.916072,87999.0,10.0,207850.0,10392.5,38260.287131,168673.0,2.0,0,…,702.892469,494057.823656,2,4,0,2,28,0,0,0,2.0,831.733333,43.111111,1169.714286,0,0,0,24952,20,402393,0,0,0,0,16383,16383,2,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Exploits"""
"""175.45.176.0-149.171.126.16-13…","""175.45.176.0""",13284,"""149.171.126.16""",80,6,"""22/01/2015 07:50:13 AM""",2376792,9,3,752.0,0.0,188.0,0.0,83.555556,99.0847,0.0,0.0,0.0,0.0,316.392852,5.048822,216072.0,638179.872769,2.138664e6,11.0,2.323484e6,290435.5,747410.857535,2.138664e6,11.0,2.357699e6,1178849.5,1.6671e6,2.35768e6,19.0,0,…,90.312279,8156.307692,2,4,0,0,10,0,0,0,0.0,62.666667,83.555556,0.0,0,0,0,0,0,0,9,752,3,0,16383,16383,4,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Reconnaissance"""
"""175.45.176.2-149.171.126.16-13…","""175.45.176.2""",13792,"""149.171.126.16""",5555,6,"""22/01/2015 07:50:16 AM""",131350,10,3,7564.0,0.0,1460.0,0.0,756.4,690.497277,0.0,0.0,0.0,0.0,57586.600685,98.972212,10945.833333,21066.252205,62942.0,2.0,119039.0,13226.555556,25983.491245,62942.0,2.0,122595.0,61297.5,86675.027918,122586.0,9.0,0,…,675.150516,455828.21978,1,4,0,0,11,0,0,0,0.0,581.846154,756.4,0.0,0,0,0,7564,6,6367003,0,0,0,0,16383,16383,6,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Exploits"""
"""175.45.176.0-149.171.126.15-39…","""175.45.176.0""",39500,"""149.171.126.15""",80,6,"""22/01/2015 07:50:18 AM""",164796,6,3,770.0,0.0,385.0,0.0,128.333333,198.813145,0.0,0.0,0.0,0.0,4672.443506,54.612976,20599.5,26851.824918,59363.0,11.0,111397.0,22279.4,30599.330537,59363.0,11.0,157703.0,78851.5,111495.890151,157691.0,12.0,0,…,162.330253,26351.111111,1,4,0,0,7,0,0,0,0.0,85.555556,128.333333,0.0,0,0,0,0,0,0,0,0,0,0,16383,16383,2,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""DoS"""
"""175.45.176.0-149.171.126.14-29…","""175.45.176.0""",29309,"""149.171.126.14""",3000,6,"""22/01/2015 07:50:19 AM""",163418,6,3,400.0,0.0,200.0,0.0,66.666667,103.279556,0.0,0.0,0.0,0.0,2447.710778,55.073493,20427.25,27063.749406,62193.0,11.0,112374.0,22474.8,31050.388568,62193.0,11.0,157512.0,78756.0,111359.418542,157499.0,13.0,0,…,84.327404,7111.111111,1,4,0,0,7,0,0,0,0.0,44.444444,66.666667,0.0,0,0,0,0,0,0,0,0,0,0,16383,16383,2,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Generic"""


In [8]:
start_time = datetime.now()
print("Converting Polars DataFrame to Pandas DataFrame...")

try:
    df = df_pl.to_pandas()
    print("Conversion completed in", datetime.now() - start_time)
except Exception as e:
    print("Conversion failed:", e)

Converting Polars DataFrame to Pandas DataFrame...
Conversion completed in 0:00:01.331845


In [9]:
print('Cantidad de Filas y columnas:',df.shape)
print('Nombre columnas:',df.columns)

Cantidad de Filas y columnas: (3480335, 84)
Nombre columnas: Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 

In [14]:
# get attack labels and print them
attack_labels = df['Label'].unique()
print(f'Unique attack labels in dataset: {sorted(attack_labels)}')

# remove 'Injection' class
df = df[~df['Label'].isin(['Backdoor', 'Analysis', 'Worms'])]
print("Data shape:", df.shape)
print("Data columns:", df.columns)

print("Encoding attack types...")
le = LabelEncoder()
df['Attack Number'] = le.fit_transform(df['Label'])
print("Encoded attack classes:")
for val in sorted(df['Attack Number'].unique()):
    print(f"{val}: {le.inverse_transform([val])[0]}")

print("Number of samples of each attack type:")
print(df['Label'].value_counts())

print("Reducing memory usage...")
old_memory_usage = df.memory_usage().sum() / 1024 ** 2
print(f'Initial memory usage: {old_memory_usage:.2f} MB')
for col in df.columns:
    col_type = df[col].dtype
    if col_type != object:
        c_min = df[col].min()
        c_max = df[col].max()
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            df[col] = df[col].astype(np.float32)
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            df[col] = df[col].astype(np.int32)
new_memory_usage = df.memory_usage().sum() / 1024 ** 2
print(f"Final memory usage: {new_memory_usage:.2f} MB")
print(f'Reduced memory usage: {1 - (new_memory_usage / old_memory_usage):.2%}')

data = df.copy()

attacks = data['Label']  # Using original Label column instead of Attack Type
data = data.select_dtypes(include=['float32', 'int32'])

selected_columns = [
    'Dst Port', 'Protocol', 'Flow Duration',
    'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Mean',
       'Fwd IAT Std', 'Fwd IAT Max', 'Bwd IAT Std', 'Bwd IAT Max',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'PSH Flag Count', 'ACK Flag Count', 'Down/Up Ratio',
       'Average Packet Size', 'Bwd Segment Size Avg', 'FWD Init Win Bytes',
       'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'
]
data = data[selected_columns]

#change the column names

data.columns = ['Destination Port', 'Protocol', 'Flow Duration',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Fwd IAT Total', 'Fwd IAT Mean',
       'Fwd IAT Std', 'Fwd IAT Max', 'Bwd IAT Std', 'Bwd IAT Max',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'PSH Flag Count', 'ACK Flag Count', 'Down/Up Ratio',
       'Average Packet Size', 'Avg Bwd Segment Size', 'Init_Win_bytes_forward',
       'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']

print("Standardizing features...")
scaler = StandardScaler()
scaled_features = scaler.fit_transform(data)

Unique attack labels in dataset: ['Benign', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode']
Data shape: (3479252, 85)
Data columns: Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Head

In [18]:
# save selected features
new_data_k_20 = data.copy()
new_data_k_20['Attack Type'] = attacks.values

# get attack type counts
attack_counts = new_data_k_20['Attack Type'].value_counts()
print("Attack type counts:")
print(attack_counts)
print(new_data_k_20.head())

Attack type counts:
Attack Type
Benign            3390752
Exploits            30951
Fuzzers             29613
Reconnaissance      16735
Generic              4632
DoS                  4467
Shellcode            2102
Name: count, dtype: int64
   Destination Port  Protocol  Flow Duration  Bwd Packet Length Max  \
0                80         6         214392                 1460.0   
1                80         6        2376792                    0.0   
2              5555         6         131350                    0.0   
3                80         6         164796                    0.0   
4              3000         6         163418                    0.0   

   Bwd Packet Length Min  Bwd Packet Length Mean  Bwd Packet Length Std  \
0                    0.0             1169.714233             552.156982   
1                    0.0                0.000000               0.000000   
2                    0.0                0.000000               0.000000   
3                    0.0         

In [19]:
from imblearn.over_sampling import SMOTE

#######################################################################

class_counts = new_data_k_20['Attack Type'].value_counts()
selected_classes = class_counts[class_counts > 1950]
class_names = selected_classes.index
selected = new_data_k_20[new_data_k_20['Attack Type'].isin(class_names)]

dfs = []
for name in class_names:
  df = selected[selected['Attack Type'] == name]
  if len(df) > 2500:
    df = df.sample(n = 2000, random_state = 0)

  dfs.append(df)

df = pd.concat(dfs, ignore_index = True)

X = df.drop('Attack Type', axis=1)
y = df['Attack Type']

smote = SMOTE(sampling_strategy='auto', random_state=0)
X_upsampled, y_upsampled = smote.fit_resample(X, y)

blnc_data = pd.DataFrame(X_upsampled)
blnc_data['Attack Type'] = y_upsampled
blnc_data = blnc_data.sample(frac=1)

features = blnc_data.drop('Attack Type', axis = 1)
labels = blnc_data['Attack Type']

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size = 0.3, random_state = 0)

In [20]:
# save X_train, X_test, y_train, y_test in a variable
data_split = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test
}
# save the data_split variable to a file
import pickle
with open('/mnt/AI-DATA/alara/CiberIA_O1_A1/Framework/data_split_unsw.pkl', 'wb') as f:
    pickle.dump(data_split, f)

In [33]:
# load data_split from 2017 and 2018 and combine into new X_train, X_test, y_train, y_test using 70% and 30% of the data
with open('/mnt/AI-DATA/alara/CiberIA_O1_A1/Framework/data_split_2017.pkl', 'rb') as f:
    data_split_2017 = pickle.load(f)
with open('/mnt/AI-DATA/alara/CiberIA_O1_A1/Framework/data_split_2018.pkl', 'rb') as f:
    data_split_2018 = pickle.load(f)


percentage_2017 = 0.9

# Get number of samples for 70% from 2017 and 30% from 2018
n_train_2017 = int(len(data_split_2017['X_train']) * percentage_2017)
n_train_2018 = len(data_split_2017['X_train']) - n_train_2017

n_test_2017 = int(len(data_split_2017['X_test']) * percentage_2017)
n_test_2018 = len(data_split_2017['X_test']) - n_test_2017

# Sample the data
X_train = pd.concat([
    data_split_2017['X_train'].sample(n=n_train_2017, random_state=0),
    data_split_2018['X_train'].sample(n=n_train_2018, random_state=0)
], ignore_index=True)
y_train = pd.concat([
    data_split_2017['y_train'].sample(n=n_train_2017, random_state=0),
    data_split_2018['y_train'].sample(n=n_train_2018, random_state=0)
], ignore_index=True)

X_test = pd.concat([
    data_split_2017['X_test'].sample(n=n_test_2017, random_state=0),
    data_split_2018['X_test'].sample(n=n_test_2018, random_state=0)
], ignore_index=True)
y_test = pd.concat([
    data_split_2017['y_test'].sample(n=n_test_2017, random_state=0),
    data_split_2018['y_test'].sample(n=n_test_2018, random_state=0)
], ignore_index=True)

In [ ]:
import joblib
#load model from a file
model_filename = '/mnt/AI-DATA/alara/CiberIA_O1_A1/Framework/stacked_model_original.pkl'
stacked_clf = joblib.load(model_filename)
print("Model loaded successfully.")
print(stacked_clf)

Model loaded successfully.
Pipeline(steps=[('scaler', StandardScaler()),
                ('stacking',
                 StackingClassifier(cv=5,
                                    estimators=[('rf',
                                                 RandomForestClassifier(max_depth=15,
                                                                        n_estimators=200)),
                                                ('lgb',
                                                 LGBMClassifier(bagging_fraction=0.8,
                                                                bagging_freq=5,
                                                                feature_fraction=0.8,
                                                                learning_rate=0.05,
                                                                max_depth=15,
                                                                min_data_in_leaf=30,
                                                                min_

In [35]:
#get accuracy using the model stacked clf and the Y test set
accuracy = stacked_clf.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.2f}")

Model accuracy: 0.98


/mnt/AI-DATA/alara/CiberIA_O1_A1/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [36]:
import joblib
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb

# Threshold below which we consider accuracy to have “dropped”
ACCURACY_THRESHOLD = 0.80

# 1. Load the existing stacked model
model_filename = '/mnt/AI-DATA/alara/CiberIA_O1_A1/Framework/stacked_model.pkl'
stacked_clf = joblib.load(model_filename)
print("Model loaded successfully.")
print(stacked_clf)

# 2. Evaluate on the new test set
accuracy = stacked_clf.score(X_test, y_test)
print(f"Model accuracy on new data: {accuracy:.2f}")

# 3. If accuracy drops below threshold, retrain on X_train, y_train
if accuracy < ACCURACY_THRESHOLD:
    print(f"Accuracy {accuracy:.2f} < {ACCURACY_THRESHOLD:.2f}. Retraining model...")

    params = {
    'min_gain_to_split': 0.1,  # Lowering the threshold for splits
    'min_child_samples': 20,   # Preventing overfitting by controlling sample count per leaf
    'min_data_in_leaf': 30,    # Ensures enough data per leaf
    'num_leaves': 40,
    'max_depth': 15,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,  # Using a fraction of features to prevent overfitting
    'bagging_fraction': 0.8,  # Using a fraction of data to prevent overfitting
    'bagging_freq': 5,  # Perform bagging every iteration
    'random_state': 0,
    'verbose': -1
    }

    # a) Recreate the stacking pipeline (replace with your original base/stacker models)
    base_learners = [
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, criterion='gini', max_features='sqrt')),
        ('lgb', lgb.LGBMClassifier(**params))
    ]
    lr = LogisticRegression(max_iter=20000, C=1.0, penalty='l2', solver='lbfgs', random_state=0)
    meta_learner = lr
    
    # b) Build a new pipeline including scaling + stacking
    #    (Assumes X_train is already the appropriate feature set)
    scaler = StandardScaler()
    stacking = StackingClassifier(
        estimators=base_learners,
        final_estimator=meta_learner,
        cv=5,
        n_jobs=-1,
        passthrough=True
    )

    # Combine scaling and stacking into a single Pipeline
    retrain_pipeline = Pipeline([
        ('scaler', scaler),
        ('stacking', stacking)
    ])

    # c) Fit on the new(train) data
    retrain_pipeline.fit(X_train, y_train)
    new_accuracy = retrain_pipeline.score(X_test, y_test)
    print(f"Retrained model accuracy: {new_accuracy:.2f}")

    # d) Save the newly trained model back to disk
    joblib.dump(retrain_pipeline, model_filename)
    print(f"Retrained model saved to {model_filename}")

else:
    print("Accuracy is above threshold—no retraining needed.")


Model loaded successfully.
Pipeline(steps=[('scaler', StandardScaler()),
                ('stacking',
                 StackingClassifier(cv=5,
                                    estimators=[('rf',
                                                 RandomForestClassifier(max_depth=15,
                                                                        n_estimators=200)),
                                                ('lgb',
                                                 LGBMClassifier(bagging_fraction=0.8,
                                                                bagging_freq=5,
                                                                feature_fraction=0.8,
                                                                learning_rate=0.05,
                                                                max_depth=15,
                                                                min_data_in_leaf=30,
                                                                min_

/mnt/AI-DATA/alara/CiberIA_O1_A1/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
